# 05 — SQL/PostgreSQL — GUIA VIVO — COLAB**MATERIAL DIDÁTICO FICTÍCIO/SINTÉTICO, NÃO OFICIAL; prévia didática revisada em 24/11/2026.** Fonte de temas: transcrições e apresentações SC TEC T8 Semanas 08–11, com divergências documentadas. Os dados são inventados; não são bases da professora ou do mini-projeto da Semana 7.**Como executar:** ambiente Google Colab (Python online; runtime temporário). Execute células Python de cima para baixo. As células SQL usam **SQLite em memória como simulação conceitual**, não PostgreSQL real. As células Mongo usam **objetos Python**, não mongosh/Compass. Use os scripts `labs/postgresql/` e `labs/mongodb/` somente em servidores de treino autorizados.Acesse separadamente `docs/ambientacao/fontes-oficiais-bancos.md`, `primeiro-acesso-postgresql.md` e `primeiro-acesso-mongodb.md` para downloads, cadastro de conexão e reconhecimento da interface. **SQL é a linguagem de PostgreSQL; mongosh usa JavaScript, diferente de Java; Python/Java são integrações por drivers, não pré-condição de uso da GUI.**CHECKPOINT: COMPREENDER (explicar), EXECUTAR (prever/rodar em ambiente correto), DIAGNOSTICAR (corrigir interpretação).

**Estado de estudo — 24/11/2026:** extensão didática das semanas 07–11. As versões e cortes históricos abaixo são preservados como histórico; os novos blocos de oficina complementam o percurso. Conteúdo autoral sintético, não oficial. Validação técnica não equivale a homologação ou progresso institucional.


**Quer executar no PostgreSQL/pgAdmin real?** Este notebook ensina conceitos com exemplos Python/SQLite. Para abrir pgAdmin, criar banco, carregar SQL, importar sete CSVs e conferir consultas, siga o [passo a passo PostgreSQL/pgAdmin](https://github.com/otavio-diniz/sc-tec-analise-dados-python/blob/main/docs/ambientacao/primeiro-acesso-postgresql.md). Os arquivos .sql/.psql estão em labs/postgresql; .psql é exclusivo do psql.

## 🔁 Você já viu isso

Este eixo reaproveita listas/dicionários, operações, Pandas, tipos, filtros e qualidade. Não trate coincidência de nome como equivalência de esquema.

In [ ]:
import sqlite3
cn=sqlite3.connect(":memory:")
cn.row_factory=sqlite3.Row
cn.executescript("""
PRAGMA foreign_keys=ON;
CREATE TABLE clientes(id INTEGER PRIMARY KEY,nome TEXT,estado TEXT);
CREATE TABLE produtos(id TEXT PRIMARY KEY,nome TEXT,preco NUMERIC);
CREATE TABLE pedidos(id INTEGER PRIMARY KEY,cliente_id INTEGER REFERENCES clientes(id),dia TEXT,status TEXT);
CREATE TABLE itens(pedido_id INTEGER REFERENCES pedidos(id),produto_id TEXT REFERENCES produtos(id),qtd INTEGER,preco NUMERIC,desconto NUMERIC,PRIMARY KEY(pedido_id,produto_id));
INSERT INTO clientes VALUES(1,'Lia','SP'),(2,'Raul','RJ'),(3,'Bia','MG');
INSERT INTO produtos VALUES('P01','Caderno',10),('P02','Caneta',20);
INSERT INTO pedidos VALUES(101,1,'2026-11-05','concluido'),(102,2,'2026-11-09','concluido');
INSERT INTO itens VALUES(101,'P01',2,10,0),(101,'P02',1,20,0),(102,'P01',3,5,10);
""")
print("Simulação SQLite na memória — NÃO é PostgreSQL conectado.")
assert cn.execute("SELECT COUNT(*) FROM clientes").fetchone()[0]==3

## Comece aqui — percurso das aulas reais de 27/10 a 19/11

**[AULA + COMPLEMENTO AUTORAL]** Na Semana 08 a professora começou por HTML, BeautifulSoup, scraping responsável, funções e **ETL → SQLite**. PostgreSQL/pgAdmin vieram depois. Na Semana 10 houve normalização e consultas; em 17 e 19/11, JOINs, subconsultas, CTE e relatórios. O material está organizado por dependência cognitiva, não pela agenda dos slides.

**Rotina inicial sem instalações:** (1) execute a célula de ETL local abaixo; (2) explique o que entrou, saiu e foi transformado; (3) siga o exemplo SQL com SQLite em memória; (4) leia [PostgreSQL/pgAdmin: primeiro acesso](../../docs/ambientacao/primeiro-acesso-postgresql.md); (5) somente num servidor PostgreSQL de treino autorizado, execute os [SQL autorais](../../labs/postgresql/00_dados_sinteticos.sql) e depois os [relatórios](../../labs/postgresql/01_consultas_e_relatorios.sql).

**Termos distintos:** PostgreSQL é servidor; pgAdmin é cliente visual; Query Tool é o editor de comandos no cliente; SQL é linguagem; SQLite é **outro banco**, usado para aprender sem conexão; Colab/VS Code executam **Python**. Ter aplicativo aberto não comprova conexão.

[Sequência das 11 aulas](../../docs/PERCURSO_SEMANAS_08_A_11.md) · [Fontes oficiais](../../docs/ambientacao/fontes-oficiais-bancos.md).

### Oficina 0 — Extrair, transformar e carregar: HTML local → tabela SQLite

**[AULA 27/10; DEMONSTRAÇÃO AUTORAL SINTÉTICA]** O site, os produtos e os preços abaixo são fictícios. Não há coleta de terceiros nem rede externa. O parser padrão de Python encontra `article.produto`, transforma preço brasileiro em **centavos inteiros** e carrega uma tabela SQLite temporária.

O que a professora mostrou com BeautifulSoup também pode ser entendido assim: `BeautifulSoup(html, "html.parser").select("article.produto")` localizaria os mesmos elementos se o pacote estivesse disponível. `requests.get` exigiria endereço permitido, limite de carga, tratamento do HTTP e autorização conforme o caso; não é necessário neste treino.

**Previsão:** 2 linhas (produtos) e total de R$ 40,00. Explique EXTRACT, TRANSFORM e LOAD antes de executar. Para estudar cada etapa: [Ponte da Semana 08](../../docs/ambientacao/ponte-semana-08-etl.md).

In [ ]:
from html.parser import HTMLParser
from decimal import Decimal
import sqlite3

class Cards(HTMLParser):
    def __init__(self):
        super().__init__()
        self.dados = []
    def handle_starttag(self, tag, attrs):
        a = dict(attrs)
        if tag == "article" and a.get("class") == "produto":
            self.dados.append(a)

html_local = """
<article class="produto" data-codigo="P01" data-preco="10,00" data-qtd="2"></article>
<article class="produto" data-codigo="P02" data-preco="20,00" data-qtd="1"></article>
"""
parser = Cards()
parser.feed(html_local)  # EXTRACT: ler HTML inventado
limpos = [(d["data-codigo"],
           int(Decimal(d["data-preco"].replace(",", "."))*100),
           int(d["data-qtd"])) for d in parser.dados]  # TRANSFORM
assert len(limpos) == 2 and all(x[2] > 0 for x in limpos)
conexao_etl = sqlite3.connect(":memory:")
conexao_etl.execute("CREATE TABLE cards(codigo TEXT PRIMARY KEY, preco_centavos INT, quantidade INT)")
conexao_etl.executemany("INSERT INTO cards VALUES (?,?,?)", limpos)  # LOAD
linhas, total = conexao_etl.execute(
    "SELECT COUNT(*), SUM(preco_centavos*quantidade) FROM cards"
).fetchone()
assert (linhas, total) == (2, 4000)
print(f"ETL validado: {linhas} produtos, R$ {total/100:.2f}")
conexao_etl.close()

## ✅ CHECKPOINT 1 — Servidor, cliente e linguagens

[AULA M1S08] PostgreSQL é o servidor; pgAdmin é o cliente gráfico; Query Tool envia SQL. O editor VS Code e o Colab executam Python em seus próprios ambientes, não criam automaticamente um servidor PostgreSQL. Primeiro contato: identificar host, porta, banco, usuário e conexão autorizada. O laboratório real deve começar em pgAdmin com `SELECT current_database();` e `SELECT current_user;`. Porta usual 5432 não é garantia de configuração.

In [ ]:
print('SQLite local:',cn.execute('SELECT 1').fetchone()[0]);assert cn.execute('SELECT 1').fetchone()[0]==1

### Domínio CP1

COMPREENDER a distinção servidor/cliente. EXECUTAR o exemplo SQLite e, futuramente, a consulta PostgreSQL no banco autorizado. DIAGNOSTICAR erro de conexão sem atribuí-lo ao SQL.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 2 — Tabelas, chaves e consultas de leitura

[AULA M1S08–10] Tabela tem linhas/colunas. PK identifica registro e FK preserva referência. `SELECT nome FROM clientes WHERE estado='SP' ORDER BY nome` significa escolher coluna, origem, filtro e ordenação. Não confunda índice de Pandas com PK de banco. No servidor real, usar schema e banco corretos.

### Da planilha repetida ao modelo normalizado (aula de 12/11)

Considere uma venda bruta com `pedido_id, cliente_id, cliente_nome, produto_id, produto_nome, quantidade`. Uma linha é **item do pedido**, não pedido completo. Identifique as repetições antes de modelar.

- **1FN:** cada coluna representa um valor atômico; não coloque vários produtos numa lista textual na coluna de pedido.
- **2FN:** com chave composta `(pedido_id, produto_id)`, atributos de cliente/pedido ou produto não devem ficar dependentes de apenas parte da chave na tabela de itens.
- **3FN:** `cliente_nome` depende de `cliente_id`, não do identificador do pedido; mantenha o cadastro numa tabela de clientes.
- **Modelo:** `clientes(id PK) → pedidos(cliente_id FK) → itens(pedido_id FK, produto_id FK) ← produtos(id PK)`. Os nomes e as restrições são de exemplo; o desenho concreto depende da fonte e das regras de negócio.

No pgAdmin de **treino**, após criar as tabelas por SQL autoral, confira a PK e as FK em `Schemas → sctec_treino → Tables`, faça `SELECT` e reconstrua os campos com `JOIN`. Treine `WHERE`, `AND/OR`, `IN`, `BETWEEN`, `ORDER BY` e `LIKE`; `ILIKE` é específico de PostgreSQL e não funciona do mesmo modo em SQLite.

**Armadilha:** preço de catálogo do produto pode diferir do preço registrado no item na data da compra; não é prova de erro de integridade.

In [ ]:
rows=cn.execute("SELECT nome FROM clientes WHERE estado='SP'").fetchall();print([r['nome'] for r in rows]);assert rows[0]['nome']=='Lia'

### Domínio CP2

COMPREENDER PK e FK. EXECUTAR SELECT com filtro. DIAGNOSTICAR tabela ausente em banco/esquema errado.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 3 — JOIN e granularidade

[AULA M1S10–11] Em JOIN pedido→itens cada item resulta uma linha. INNER exige par; LEFT preserva linhas da esquerda. Cliente Bia não possui pedido. `COUNT(*)` após JOIN dá 3 linhas; `COUNT(DISTINCT pedido_id)` dá 2 pedidos. WHERE sobre coluna de uma tabela opcional pode eliminar as linhas preservadas pelo LEFT. RIGHT/FULL são mostrados na fonte SQL PostgreSQL, mas não homologados aqui em SQLite.

### INNER, LEFT, RIGHT e FULL — antecipe a granularidade

`INNER JOIN` mantém correspondências. `LEFT JOIN` preserva toda a tabela da esquerda (por exemplo, Bia sem pedido); `RIGHT JOIN` preserva toda a direita; `FULL OUTER JOIN` preserva ausências de ambos os lados. No PostgreSQL de treino:
`SELECT c.nome,p.id FROM sctec_treino.pedidos p RIGHT JOIN sctec_treino.clientes c ON p.cliente_id=c.id;`
`SELECT c.nome,p.id FROM sctec_treino.clientes c FULL JOIN sctec_treino.pedidos p ON c.id=p.cliente_id;`

O `JOIN` pedido×item dá **3 linhas**, embora existam **2 pedidos**; `COUNT(DISTINCT p.id)` protege a contagem de pedidos. `WHERE` aplicado à tabela opcional após `LEFT JOIN` pode eliminar os nulos que você queria preservar. `RIGHT/FULL` foram exercitados na aula avançada e são detalhados no script PostgreSQL (não na simulação SQLite).

In [ ]:
r=cn.execute('SELECT COUNT(*) a, COUNT(DISTINCT p.id) b FROM pedidos p JOIN itens i ON i.pedido_id=p.id').fetchone();assert (r['a'],r['b'])==(3,2);print(dict(r))

### Domínio CP3

COMPREENDER 1:N. EXECUTAR JOIN com previsão. DIAGNOSTICAR duplicação de indicadores.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 4 — GROUP BY, HAVING e métricas

[AULA M1S11] WHERE filtra linhas; GROUP BY organiza grupos; HAVING filtra os grupos. Pedidos, unidades e valor de itens são medidas distintas. `qtd * preco * (1 - desconto/100)` produz valor didático, não lucro ou pagamento recebido. Valor esperado: pedido 101 R$40; pedido 102 R$13,50; total R$53,50 e 6 unidades.

### Consultas de negócio — agregação e relatórios

`WHERE` filtra **linhas**; `GROUP BY` define grupos; `HAVING` filtra **grupos**; `COUNT(*)` conta linhas e `COUNT(DISTINCT pedido_id)` conta pedidos. `SUM(qtd)` soma unidades. `UNION ALL` empilha resultados **sem eliminar duplicatas**; não confundir com `UNION`. A instrutora também usou `RIGHT/FULL`, concatenação e funções de texto em 19/11.

Na amostra, os itens somam R$ 53,50 **após descontos** — isso **não é lucro, dinheiro recebido ou receita fiscal**. Sempre fixe unidade de análise, período e status do pedido antes da métrica.

In [ ]:
r=cn.execute('SELECT COUNT(DISTINCT pedido_id) n,SUM(qtd) u,SUM(qtd*preco*(1-desconto/100.0)) v FROM itens').fetchone();assert (r['n'],r['u'],round(r['v'],2))==(2,6,53.5);print(dict(r))

### Domínio CP4

COMPREENDER unidade de análise e filtro de status. EXECUTAR soma por pedido. DIAGNOSTICAR receita inflada por JOIN.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 5 — Subconsultas e condições de negócio

[AULA M1S11] Subconsulta pode devolver valor escalar, conjunto IN ou tabela derivada com alias. Acima da média global NÃO prova outlier quando categorias diferem. Cliente com UM pedido acima de um limite é diferente de cliente cuja SOMA supera o limite. Primeiro escreva a pergunta de negócio em linguagem natural.

### Subconsultas → CTE → datas, sem misturar os dialetos

A aula de 19/11 distingue subconsulta **escalar** (`AVG`), **conjunto** (`IN`) e **tabela derivada no FROM`. A CTE (`WITH`) nomeia uma etapa intermediária e facilita leitura, mas não garante mais velocidade nem cria tabela física persistente.

No PostgreSQL, `DATE_TRUNC('month', dia)::date` identifica o início do mês para agrupamento. `CONCAT` monta rótulo e `SUBSTRING` extrai parte do texto. Na simulação Python/SQLite a célula usa `substr`, **não o dialeto PostgreSQL**.

**Cuidado que a professora explicitou:** comparar novembro **parcial** com outubro completo gera interpretação enganosa. Um produto com preço acima da média global também **não é automaticamente um outlier**.

In [ ]:
q='SELECT pedido_id FROM (SELECT pedido_id,SUM(qtd*preco*(1-desconto/100.0)) total FROM itens GROUP BY pedido_id) r WHERE total>20';rs=cn.execute(q).fetchall();assert [x[0] for x in rs]==[101];print('Pedidos >20:',[x[0] for x in rs])

### Domínio CP5

COMPREENDER limiar por pedido vs total cliente. EXECUTAR tabela derivada. DIAGNOSTICAR média fora de contexto.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 6 — CTE e legibilidade

[AULA M1S11] WITH nome AS (...) nomeia uma consulta intermediária e facilita decompor um relatório; não garante aceleração por si. A CTE deve preservar a granularidade (um resumo por pedido). Um JOIN posterior pode voltar a multiplicar linhas se for 1:N.

In [ ]:
q='WITH r AS (SELECT pedido_id,SUM(qtd) unidades FROM itens GROUP BY pedido_id) SELECT COUNT(*) FROM r';assert cn.execute(q).fetchone()[0]==2;print('CTE: dois resumos de pedido')

### Domínio CP6

COMPREENDER a finalidade do WITH. EXECUTAR resumo e previsão. DIAGNOSTICAR perda de granularidade.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 7 — Datas e interpretação temporal

[AULA M1S11] PostgreSQL possui `DATE_TRUNC('month', data_pedido)::date`, usado para agrupamento por mês; não é comando SQLite. O exemplo SQLite usa substring textual SOMENTE para ilustrar o mês. Comparar novembro parcial com outubro completo sem ajustar janelas induz conclusão errada. CONCAT e SUBSTRING organizam apresentação, não validam cadastros.

In [ ]:
r=cn.execute('SELECT substr(dia,1,7) mes,COUNT(*) n FROM pedidos GROUP BY substr(dia,1,7)').fetchall();print([tuple(x) for x in r]);assert len(r)==1

### Domínio CP7

COMPREENDER mês parcial. EXECUTAR previsão de agrupamento. DIAGNOSTICAR diferenças de dialeto SQL.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 8 — Transações, integridade e comunicação

[AULA M1S08–11; COMPLEMENTO] Antes de escrever dados: confirmar conexão, isolamento, PK/FK, status, quantidade de linhas afetadas e reversibilidade. PostgreSQL suporta transações BEGIN/ROLLBACK em muitos casos; isto não torna qualquer ação inofensiva. Nem o notebook nem a presença do SQL provam sua execução em servidor real. Não usar scripts da professora em base de produção.

### Antes de alterar dados no servidor real

Confirme banco, schema, conta e permissão, rode um `SELECT` com o filtro pretendido, preveja quantas linhas serão afetadas, use `BEGIN` e `ROLLBACK/COMMIT` quando adequado, e valide PK/FK/`DEFAULT`/`CHECK`. Instalação ou conexão nunca autorizam modificar bases existentes da professora. As consultas deste notebook **não alteram PostgreSQL real**.

In [ ]:
assert cn.execute('PRAGMA foreign_key_check').fetchall()==[];print('Sem violações de FK nos dados fictícios SQLite')

### Domínio CP8

COMPREENDER integridade e escopo. EXECUTAR checagens. DIAGNOSTICAR consulta correta com interpretação comercial incorreta.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## Revisão acumulativa

Explique, sem consultar: servidor vs cliente, sintaxe da linguagem vs formato de dados, documentos vs linhas, aula observada vs slide planejado, exemplo sintético vs tarefa formal. A execução Python deste Guia **não valida** os scripts em PostgreSQL ou MongoDB.

**Próximo passo:** abra o Caderno Prático espelhado. As soluções aparecem após espaço de tentativa; só consulte depois de responder.

## Oficina integradora — Semanas 10 e 11: normalização e reconciliação
MATERIAL DIDÁTICO FICTÍCIO/SINTÉTICO — dados autorais, não oficiais. 1FN: valores atômicos; 2FN: dependência da chave completa; 3FN: ausência de dependência transitiva entre atributos não-chave. A tabela bruta tem uma linha por item de pedido. Verifique conflitos de cadastro antes de migrar; DISTINCT não resolve divergências. PK identifica, FK protege relacionamentos, CHECK impede valores inválidos. A célula usa SQLite em memória como modelo; pratique PostgreSQL no servidor de treino. PostgreSQL 16+ permite omitir alias de subconsulta no FROM, mas use alias explícito por clareza e compatibilidade.

In [ ]:
import sqlite3
con_norm=sqlite3.connect(':memory:')
con_norm.execute('PRAGMA foreign_keys=ON')
con_norm.executescript('''
CREATE TABLE bruto(pedido INTEGER, cliente INTEGER, nome TEXT, estado TEXT, produto INTEGER, descricao TEXT, qtd INTEGER, centavos INTEGER);
INSERT INTO bruto VALUES (71,1,'Nara','SC',11,'Agenda',2,1500),(71,1,'Nara','SC',12,'Caneta',1,500),(72,2,'Ivo','PR',11,'Agenda',1,1500);
CREATE TABLE clientes(id INTEGER PRIMARY KEY,nome TEXT,estado TEXT);
CREATE TABLE produtos(id INTEGER PRIMARY KEY,descricao TEXT);
CREATE TABLE pedidos(id INTEGER PRIMARY KEY,cliente_id INTEGER REFERENCES clientes(id));
CREATE TABLE itens(pedido_id INTEGER REFERENCES pedidos(id),produto_id INTEGER REFERENCES produtos(id),qtd INTEGER CHECK(qtd>0),centavos INTEGER CHECK(centavos>=0),PRIMARY KEY(pedido_id,produto_id));
''')
assert not con_norm.execute('SELECT cliente FROM bruto GROUP BY cliente HAVING COUNT(DISTINCT nome)>1 OR COUNT(DISTINCT estado)>1').fetchall()
assert not con_norm.execute('SELECT produto FROM bruto GROUP BY produto HAVING COUNT(DISTINCT descricao)>1').fetchall()
assert not con_norm.execute('SELECT pedido FROM bruto GROUP BY pedido HAVING COUNT(DISTINCT cliente)>1').fetchall()
con_norm.executescript('''
INSERT INTO clientes SELECT DISTINCT cliente,nome,estado FROM bruto;
INSERT INTO produtos SELECT DISTINCT produto,descricao FROM bruto;
INSERT INTO pedidos SELECT DISTINCT pedido,cliente FROM bruto;
INSERT INTO itens SELECT pedido,produto,qtd,centavos FROM bruto;
''')
antes=con_norm.execute('SELECT SUM(qtd*centavos) FROM bruto').fetchone()[0]
depois=con_norm.execute('SELECT SUM(i.qtd*i.centavos) FROM itens i JOIN pedidos p ON p.id=i.pedido_id JOIN clientes c ON c.id=p.cliente_id JOIN produtos pr ON pr.id=i.produto_id').fetchone()[0]
assert antes==depois==5000
assert con_norm.execute('SELECT COUNT(*),COUNT(DISTINCT pedido_id) FROM itens').fetchone()==(3,2)
print('Total preservado em centavos:',depois)
con_norm.close()

**Domínio:** explique por que 3 itens não são 3 pedidos. Planeje relatório por vendedor e mês: agregue vendas e metas separadamente na mesma granularidade antes do JOIN; preserve vendedores sem movimento com LEFT JOIN. Não confunda preço acima da média com outlier. Compare períodos completos e parciais.

## Oficina Semanas 10–11 — metas, granularidade e transações
**Treino sintético.** Uma meta tem granularidade vendedor/mês; um item pertence a um pedido; não some metas depois de JOIN com itens. Agregue pedidos válidos antes de unir meta. Cadastro sem meta e meta sem cadastro são exceções distintas; em PostgreSQL use FULL OUTER JOIN com filtro no mês antes da junção. A meta importada pode ficar em tabela de estágio sem FK para diagnosticar órfãos. NULL significa ausência de meta, não meta zero. DDL CREATE, ALTER e restrições; DML INSERT/UPDATE/DELETE. BEGIN/COMMIT/ROLLBACK delimitam transações; após erro PostgreSQL, rollback ou rollback to savepoint antes de prosseguir. SQLite abaixo simula o raciocínio, não substitui o dialeto PostgreSQL. Consulte o laboratório SQL 02 para DATE_TRUNC, ILIKE, FULL e checagens reais. Importação CSV: conferir cabeçalho, tipos, ordem de dependências e contagens; usar clientes/vendedores/produtos antes de pedidos e itens. Não misturar mês parcial e completo.

In [ ]:
import sqlite3
with sqlite3.connect(":memory:") as db:
    db.execute("PRAGMA foreign_keys=ON")
    db.executescript("""CREATE TABLE vendedor(id INTEGER PRIMARY KEY,nome TEXT);
    CREATE TABLE meta(id INTEGER,mes TEXT,valor INTEGER CHECK(valor>0),PRIMARY KEY(id,mes));
    CREATE TABLE pedido(id INTEGER PRIMARY KEY,vendedor INTEGER REFERENCES vendedor(id),dia TEXT,status TEXT);
    CREATE TABLE item(pedido INTEGER REFERENCES pedido(id),sku TEXT,qtd INTEGER CHECK(qtd>0),preco INTEGER CHECK(preco>=0),desconto INTEGER CHECK(desconto BETWEEN 0 AND 100),PRIMARY KEY(pedido,sku));
    INSERT INTO vendedor VALUES(1,'Lina'),(2,'Davi'),(3,'Iara');
    INSERT INTO meta VALUES(1,'2026-10',8000),(2,'2026-10',4000),(9,'2026-10',1000);
    INSERT INTO pedido VALUES(51,1,'2026-10-03','concluido'),(52,1,'2026-10-20','concluido'),(53,2,'2026-10-25','cancelado'),(54,2,'2026-11-05','concluido');
    INSERT INTO item VALUES(51,'Z1',2,2000,0),(51,'Z2',1,1000,0),(52,'Z3',1,4000,25),(53,'Z1',1,2000,0),(54,'Z3',1,4000,0);""")
    q = """WITH realizado AS (
    SELECT p.vendedor, SUM(i.qtd*i.preco*(100-i.desconto)/100) valor
    FROM pedido p JOIN item i ON i.pedido=p.id
    WHERE p.status NOT IN ('cancelado','devolvido') AND substr(p.dia,1,7)='2026-10'
    GROUP BY p.vendedor)
    SELECT v.nome,COALESCE(r.valor,0),m.valor FROM vendedor v
    LEFT JOIN realizado r ON v.id=r.vendedor
    LEFT JOIN meta m ON v.id=m.id AND m.mes='2026-10' ORDER BY v.id"""
    assert db.execute(q).fetchall()==[('Lina',8000,8000),('Davi',0,4000),('Iara',0,None)]
    assert db.execute("SELECT m.id FROM meta m LEFT JOIN vendedor v ON m.id=v.id WHERE v.id IS NULL").fetchall()==[(9,)]
    db.execute("SAVEPOINT tentativa")
    try:
        db.execute("INSERT INTO item VALUES(52,'Z4',0,300,0)")
        raise AssertionError("CHECK ausente")
    except sqlite3.IntegrityError:
        db.execute("ROLLBACK TO tentativa")
    db.execute("RELEASE tentativa")
    assert db.execute("SELECT COUNT(*) FROM item").fetchone()==(5,)
    print(db.execute(q).fetchall())


**Interpretação:** Lina atingiu 100% da meta de outubro; Davi tem zero realizado válido; Iara não tem meta; id 9 é meta órfã. Novembro contém um pedido de R$ 40,00, não é comparação mensal fechada. Explique COUNT(*) versus COUNT(DISTINCT pedido), WHERE versus HAVING, e por que a média não identifica outlier automaticamente.

## Oficina complementar — qualidade de cadastro e operações SQL (30/10 e 13/11)
**Treino sintético.** COUNT(*) conta linhas; COUNT(coluna) ignora NULL; IS NULL localiza ausências sem excluir automaticamente. Compare valor registrado do pedido e soma dos itens: uma diferença pode ser ausência de item, desconto ou erro de regra; não corrija sem evidência. CREATE DATABASE fora de BEGIN; registre conexão e confirme current_database/current_schema. Tipos: NUMERIC para dinheiro, DATE para dia, TEXT/CHAR para códigos; UNIQUE permite nulos conforme regra do motor. IDENTITY gera identificadores. AND combina condições, OR aceita uma delas, NOT nega, parênteses explicitam ordem; BETWEEN inclui limites. ALTER acrescenta coluna, DROP remove estrutura; UPDATE SET altera linhas e RETURNING retorna afetadas; DELETE exige filtro; SAVEPOINT e ROLLBACK preservam controle. Laboratórios 03 (qualidade/DDL) e 04 (CSV via psql) no repositório. No pgAdmin, importe CSV pelo assistente de tabela com cabeçalho, delimitador e codificação corretos; valide contagens antes de avançar. Não use scripts da docente em base habitual.

In [ ]:
import sqlite3
with sqlite3.connect(':memory:') as q:
    q.executescript("CREATE TABLE c(id INT PRIMARY KEY, contato TEXT); INSERT INTO c VALUES(1,'x'),(2,NULL),(3,NULL); CREATE TABLE p(id INT PRIMARY KEY,total INT); INSERT INTO p VALUES(11,4500),(12,3000),(13,1000); CREATE TABLE i(pedido INT,valor INT); INSERT INTO i VALUES(11,4000),(11,500),(12,2500);")
    assert q.execute('SELECT COUNT(*),COUNT(contato) FROM c').fetchone()==(3,1)
    r=q.execute('SELECT p.id,p.total,SUM(i.valor) FROM p LEFT JOIN i ON i.pedido=p.id GROUP BY p.id,p.total HAVING COUNT(i.pedido)=0 OR p.total<>SUM(i.valor) ORDER BY p.id').fetchall()
    assert r==[(12,3000,2500),(13,1000,None)]
    print(r)


### Complemento dos slides — não confundir com aula prática comprovada
OLTP atende operações curtas; OLAP organiza histórico para análise. Uma tabela fato de itens pode ligar dimensões cliente, produto e tempo: defina primeiro a granularidade (um item), não misture medidas de pedido. Subconsulta correlacionada usa a linha externa; VIEW guarda uma definição consultável, não necessariamente dados materializados; CASE cria categorias condicionais. Na agenda da Semana 11, psycopg2 é driver Python, cursor executa SQL parametrizado e fetch recupera resultados, commit/rollback controla a sessão; SQLAlchemy engine configura conexão e Session controla objetos, FK é regra do banco e relationship navega objetos. São referências de planejamento, não evidência de prática oral em 17–19/11 ou aula em 20/11. O laboratório SQL usa TEMP VIEW apenas como recurso de organização da demonstração.